In [20]:
import pandas as pd
import matplotlib.pyplot as plt

In [21]:
df = pd.read_csv("nba_games.csv", index_col=0)

In [22]:
missing_counts = df.isna().sum()
# Show only columns that have at least one missing value
print(missing_counts[missing_counts > 0])

ft%             1
gmsc        27332
+/-         27332
ft%_opp         1
gmsc_opp    27332
+/-_opp     27332
dtype: int64


In [23]:
df = df.sort_values("date")
df = df.reset_index(drop=True)

In [24]:
df = df.drop(columns=["mp_opp","index_opp","gmsc","+/-","gmsc_opp","+/-_opp","total","total_opp"])

In [25]:
df.loc[df["ft%"].isna(),"ft%"] = 0
df.loc[df["ft%_opp"].isna(),"ft%_opp"] = 0

In [26]:
df["target"] = df["won"].astype(int)

In [27]:
cols_to_roll = ['fg', 'fga', 'fg%', '3p', '3pa', '3p%', 'ft', 'fta', 'ft%', 'orb',
       'drb', 'trb', 'ast', 'stl', 'blk', 'tov', 'pf', 'ts%', 'efg%',
       '3par', 'ftr', 'orb%', 'drb%', 'trb%', 'ast%', 'stl%', 'blk%', 'tov%',
       'usg%', 'ortg', 'drtg', 'won', 'pts']


def compute_rolling_averages(df):
    # 1. Crucial: Sort by date so rolling averages don't look into the future!
    df = df.sort_values("date").reset_index(drop=True)
    
    # 2. Create the list of opponent columns dynamically
    opp_cols_to_roll = [f"{col}_opp" for col in cols_to_roll]
    all_cols_to_roll = cols_to_roll + opp_cols_to_roll
    
    # 3. Roll everything grouped by the main 'team'
    for col in all_cols_to_roll:
        if col in df.columns: # Safety check
            df[f"{col}_roll10"] = ( 
                df.groupby("team")[col]
                .rolling(10, closed='left')
                .mean()
                .reset_index(level=0, drop=True)
            )

    # 4. Drop original columns to prevent the model from cheating
    # Keep the target variables: 'won', 'pts', 'won_opp', 'pts_opp'
    targets = ['won', 'pts', 'won_opp', 'pts_opp']
    cols_to_drop = [c for c in all_cols_to_roll if c not in targets]
    df = df.drop(columns=cols_to_drop)
    
    # Drop rows that don't have 10 past games yet
    df = df.dropna() 

    # Grab just the Game ID, the Team name, and their rolling stats
    # We only need the base stats (their offense) and opp stats (their defense)
    rolling_stats_only = [col for col in df.columns if "_roll10" in col and "opp" not in col]
    lookup_df = df[['id', 'team'] + rolling_stats_only].copy()
    
    # Rename columns to indicate these belong to the specific opponent
    lookup_df = lookup_df.rename(columns={'team': 'team_opp'})
    for col in rolling_stats_only:
        lookup_df = lookup_df.rename(columns={col: f"{col}_opp_history"})
        
    # Merge them back into the main DataFrame matching the game 'id' and 'team_opp'
    df = df.merge(lookup_df, on=['id', 'team_opp'], how='left')
   
    df = df.dropna() 
    
    return df

In [28]:
df_rolled = compute_rolling_averages(df.copy())

In [29]:
df_rolled = df_rolled[df_rolled["home"] == 1].copy().reset_index(drop=True)
df_rolled = df_rolled.drop(columns=["home","home_opp"])

In [30]:
df_rolled["home_elo"] = 1500
df_rolled["away_elo"] = 1500
df_rolled["home_elo"] = pd.Series(dtype='float64')
df_rolled["away_elo"] = pd.Series(dtype='float64')


teams = df_rolled["team"].unique()
teams_dict = {} 
year = df_rolled.iloc[0]["season"]

def calculate_elo(home_elo, away_elo, home_score, away_score, target):
    E_home = 1 / (1 + 10**((away_elo - home_elo) / 400))
    E_away = 1 / (1 + 10**((home_elo - away_elo) / 400))

    mov = abs(home_score - away_score)

    k = 20 * ((mov + 3)**0.8) / (7.5 + 0.006 * abs(home_elo - away_elo))

    new_home_elo = k * (target - E_home) + home_elo

    away_win = 1 - target  

    new_away_elo = k * (away_win - E_away) + away_elo
    return new_home_elo, new_away_elo
    
for row in df_rolled.itertuples():

    if row.team_opp not in teams_dict:
        teams_dict[row.team_opp] = 1500

    if row.team not in teams_dict:
        teams_dict[row.team] = 1500

    # Year-to-year carryover
    if row.season > year:
        year = row.season
        for team in teams_dict:
            teams_dict[team] = teams_dict[team]*0.75 + 1505*0.25
    
    # 1. Get the current ELOs from the dictionary
    current_home_elo = teams_dict[row.team]
    current_away_elo = teams_dict[row.team_opp]
    
    # 2. Update the DataFrame with the ELOs used for the game
    df_rolled.at[row.Index, "home_elo"] = current_home_elo
    df_rolled.at[row.Index, "away_elo"] = current_away_elo
    
    # 3. Calculate new ELOs using the *current* values
    new_home_elo, new_away_elo = calculate_elo(
        current_home_elo, current_away_elo, 
        row.pts, row.pts_opp, row.target
    )

    # 4. Update the dictionary for the next game
    teams_dict[row.team] = new_home_elo
    teams_dict[row.team_opp] = new_away_elo

df_rolled.head(12)[["team","team_opp","home_elo","away_elo","target"]]


,team,team_opp,home_elo,away_elo,target
0,SAC,TOR,1500.000000,1500.000000,1
1,MEM,OKC,1500.000000,1500.000000,1
2,PHI,DAL,1500.000000,1500.000000,0
3,BRK,ATL,1500.000000,1500.000000,1
4,DET,CLE,1500.000000,1500.000000,1
5,GSW,TOR,1500.000000,1492.267272,1
6,NOP,DEN,1500.000000,1500.000000,0
7,NYK,CHO,1500.000000,1500.000000,1
8,HOU,POR,1500.000000,1500.000000,1
9,CHO,BRK,1490.920689,1504.831864,1


In [31]:
df_rolled = df_rolled.sort_values("date").reset_index(drop=True)

In [32]:
string_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
X = df_rolled.drop(columns=string_cols + ["target","pts","pts_opp","won","usg%_roll10","usg%_opp_roll10","usg%_roll10_opp_history"])
y = df_rolled["target"]

In [33]:
# Check out crossfold validation and after trying models plot results
# Also check neural net

from sklearn.model_selection import KFold, cross_val_score, train_test_split, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import numpy as np

# Split into training and validation
X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2,random_state=42,shuffle=False)

#Train model using K-Fold Validation
tscv = TimeSeriesSplit(n_splits=5)

model = RandomForestClassifier(random_state=42,n_estimators=1000)
score = np.mean(cross_val_score(model,X_train,y_train,cv=tscv,scoring="accuracy"))

# Check most important features
model.fit(X_train,y_train)
importances = model.feature_importances_

feature_info = pd.DataFrame({'Feature': X_train.columns, 'Importance': importances})
feature_info = feature_info.sort_values(by='Importance', ascending=False)
print(f"Model accuracy - Random Forest: {score:.4f}")
print("\nTop 5 Most Important Features:")
print(feature_info.head(5))

Model accuracy - Random Forest: 0.6378

Top 5 Most Important Features:
                    Feature  Importance
97                 home_elo    0.034817
98                 away_elo    0.027545
82   ts%_roll10_opp_history    0.012785
63          drtg_opp_roll10    0.012709
93  ortg_roll10_opp_history    0.012608


In [34]:
# Logistic Regression
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

logreg_model = LogisticRegression(max_iter=200,C=0.01,tol=0.001)
score = np.mean(cross_val_score(logreg_model,X_train_scaled,y_train,cv=tscv,scoring="accuracy"))
print(f"Model accuracy - Logistic Regression: {score:.4f}")

Model accuracy - Logistic Regression: 0.6459


Model accuracy - Logistic Regression: 0.6459

In [35]:
# from xgboost import XGBClassifier
# from sklearn.metrics import accuracy_score

# model = XGBClassifier(n_estimators=3000,
#     max_depth=250,           
#     learning_rate=0.1,     
#     random_state=42)

# model.fit(X_train_scaled,y_train)
# y_pred = model.predict(X_val_scaled)
# print(f"Model accuracy - XGB: {accuracy_score(y_pred,y_val)}")

In [36]:
# Final predictian logreg
from sklearn.metrics import accuracy_score
logreg_model.fit(X_train_scaled,y_train)
y_final_pred = logreg_model.predict(X_val_scaled)
accuracy = accuracy_score(y_final_pred,y_val)

print(f"Final accuracy - LogReg: {accuracy:.3f}") 

Final accuracy - LogReg: 0.653


0.653

In [37]:
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import pandas as pd

# 1. Create your new Regression Target: Margin of Victory
# Positive margin = Home Win, Negative margin = Away Win
y_margin = df_rolled['pts'] - df_rolled['pts_opp']


# Define your features and targets
y_binary = df_rolled['target']      # What we will use to check our final accuracy

# 2. Setup TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)
accuracies = []

for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train_margin, y_test_margin = y_margin.iloc[train_index], y_margin.iloc[test_index]
    y_test_binary = y_binary.iloc[test_index] 
    
    # 3. Always Scale Linear Models
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 4. Train the Ridge Regressor
    # Note: In Ridge, the parameter is 'alpha' instead of 'C'. 
    # Higher alpha = stronger regularization (less overfitting).
    ridge_model = Ridge(alpha=10.0, random_state=42)
    ridge_model.fit(X_train_scaled, y_train_margin)
    
    # Predict the actual point differential (e.g., +5.2 points, -2.1 points)
    margin_preds = ridge_model.predict(X_test_scaled)
    
    # 5. THE HACK: Convert point differential back to binary Win/Loss
    # If predicted margin > 0, we predict a Win (1). Otherwise, Loss (0).
    win_preds = (margin_preds > 0).astype(int)
    
    # 6. Calculate accuracy against the actual binary results
    fold_acc = accuracy_score(y_test_binary, win_preds)
    accuracies.append(fold_acc)

score = np.mean(accuracies)
print(f"Ridge Regression (Margin to Binary) Accuracy: {score:.4f}")

Ridge Regression (Margin to Binary) Accuracy: 0.6464


In [38]:
# Final prediction Ridge Regressor
y_final_pred = ridge_model.predict(X_val_scaled)
y_final_pred = (y_final_pred > 0).astype(int)
accuracy = accuracy_score(y_final_pred,y_val)

print(f"Final accuracy - Ridge Regression: {accuracy:.3f}") 

Final accuracy - Ridge Regression: 0.649
